In [1]:
import torch
import matplotlib.pyplot as plt

from laya.transformer import TinyTransformerEncoder
from laya.decision import TinyDecisionModel
from laya.training import train_step
from laya.contextual_rlcd import contextual_rlcd_step
from laya.data import build_decision_sequence, encode_batch

In [2]:
contexts = [
    build_decision_sequence(
        state_tokens=[
            "customer",
            "charged",
            "twice",
        ],
        question_tokens=[
            "which",
            "department",
        ],
        options=[
            "billing",
            "technical",
            "sales",
        ],
    ),
    build_decision_sequence(
        state_tokens=[
            "app",
            "crashes",
            "after",
            "login",
        ],
        question_tokens=[
            "which",
            "department",
        ],
        options=[
            "billing",
            "technical",
            "sales",
        ],
    ),
    build_decision_sequence(
        state_tokens=[
            "customer",
            "wants",
            "enterprise",
            "pricing",
        ],
        question_tokens=[
            "which",
            "department",
        ],
        options=[
            "billing",
            "technical",
            "sales",
        ],
    ),
]

special_tokens = [
    "[PAD]",
]

vocabulary = special_tokens + sorted(
    {token for sequence in contexts for token in sequence}
)

token_to_id = {token: i for i, token in enumerate(vocabulary)}

token_ids = encode_batch(
    sequences=contexts,
    token_to_id=token_to_id,
)

mask_token_id = token_to_id["[MASK]"]

option_mask = token_ids == mask_token_id

token_ids.shape, option_mask.sum(dim=-1)

(torch.Size([3, 15]), tensor([3, 3, 3]))

In [3]:
true_distributions = torch.tensor(
    [
        [0.80, 0.15, 0.05],
        [0.10, 0.80, 0.10],
        [0.10, 0.20, 0.70],
    ]
)

In [4]:
def sample_outcomes(
    true_distributions: torch.Tensor,
) -> torch.Tensor:
    return torch.multinomial(
        true_distributions,
        num_samples=1,
        replacement=True,
    ).squeeze(-1)

In [5]:
encoder = TinyTransformerEncoder(
    vocab_size=len(vocabulary),
    max_seq_len=token_ids.shape[1],
    d_model=32,
    n_heads=4,
    d_ff=64,
    n_layers=2,
    causal=False,
)
model = TinyDecisionModel(
    encoder=encoder,
    d_model=32,
    decision_hidden_dim=32,
)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [6]:
# pre train

with torch.no_grad():
    _, proba_before = model(token_ids=token_ids, option_mask=option_mask)

proba_before

tensor([[0.3339, 0.3637, 0.3023],
        [0.3405, 0.3467, 0.3128],
        [0.3418, 0.3465, 0.3117]])

In [7]:
# supervised learning
supervised_step = 1000
supervised_error_history = []

for step in range(supervised_step):
    targets = sample_outcomes(true_distributions)

    # cross entropy
    train_step(model, optimizer, token_ids, option_mask, targets)

    with torch.no_grad():
        _, proba = model(token_ids=token_ids, option_mask=option_mask)

        err = (proba - true_distributions).pow(2).mean()

        supervised_error_history.append(err.item())

In [8]:
# post train

with torch.no_grad():
    _, proba_after = model(token_ids=token_ids, option_mask=option_mask)

proba_after

tensor([[0.7153, 0.2066, 0.0781],
        [0.1011, 0.7590, 0.1398],
        [0.1013, 0.2055, 0.6932]])

In [9]:
# rlcd
rlcd_optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

rlcd_steps = 2000

rlcd_error_history = []

for step in range(rlcd_steps):
    outcomes = sample_outcomes(true_distributions)

    contextual_rlcd_step(
        model,
        rlcd_optimizer,
        token_ids,
        option_mask,
        outcomes,
        n_candidates=32,
        exploration_std=0.2,
    )

    with torch.no_grad():
        _, proba = model(token_ids=token_ids, option_mask=option_mask)
        err = (proba - true_distributions).pow(2).mean()
        rlcd_error_history.append(err.item())

In [10]:
with torch.no_grad():
    _, proba_after_rlcd = model(token_ids=token_ids, option_mask=option_mask)

proba_after_rlcd

tensor([[0.7990, 0.1497, 0.0512],
        [0.0984, 0.7943, 0.1073],
        [0.0924, 0.1991, 0.7085]])

In [ ]:
print("True distributions:")
print(true_distributions)

print()

print("After supervised:")
print(proba_after)

print()

print("After RLCD:")
print(proba_after_rlcd)

True distributions:
tensor([[0.8000, 0.1500, 0.0500],
        [0.1000, 0.8000, 0.1000],
        [0.1000, 0.2000, 0.7000]])

After supervised:
tensor([[0.7153, 0.2066, 0.0781],
        [0.1011, 0.7590, 0.1398],
        [0.1013, 0.2055, 0.6932]])

After RLCD:
tensor([[0.7990, 0.1497, 0.0512],
        [0.0984, 0.7943, 0.1073],
        [0.0924, 0.1991, 0.7085]])
